# Avant / Après : difmap brut vs wrapper Python

In [ ]:
FITS_FILE = "tests/test_data/0003-066_X.SPLIT.1"
POL       = "RR"

---
## AVANT — subprocess

Pour du traitement batch classique (charger, afficher, sauvegarder), l'approche subprocess fonctionne bien.
On peut traiter plusieurs fichiers en parallèle, c'est simple et ça suffit.

In [ ]:
import subprocess, os, re, glob
from IPython.display import Image, display

def process_file_brut(fits_file, output_dir="results"):
    os.makedirs(output_dir, exist_ok=True)
    uv_png  = f"{output_dir}/_tmp_uv.png"
    map_png = f"{output_dir}/_tmp_map.png"

    script = f"""
observe {fits_file}
select {POL}
device {uv_png}/PNG
uvplot
device {map_png}/PNG
mapsize 512,0.1
invert
mapplot
quit
"""
    out = subprocess.run(["difmap"], input=script, capture_output=True, text=True)
    source = re.search(r"Found source:\s*(\S+)", out.stdout + out.stderr).group(1)
    os.rename(uv_png,  f"{output_dir}/{source}_uvplot.png")
    os.rename(map_png, f"{output_dir}/{source}_dirtymap.png")
    print(f"{source} — OK")

# Traitement de tous les fichiers, un par un (ou en multiprocessing)
for f in sorted(glob.glob("tests/test_data/*SPLIT*")):
    process_file_brut(f)

Résultat : des PNG générés par PGPLOT, style fixe de difmap.

In [ ]:
display(Image("results/0003-066_uvplot.png", width=500))
display(Image("results/0003-066_dirtymap.png", width=500))

---
## APRÈS — ce que le wrapper permet et que subprocess ne peut pas faire

---
### 1. Accéder aux données

Avec subprocess, difmap garde les données pour lui. Impossible de récupérer les visibilités.
Avec le wrapper, elles deviennent des tableaux NumPy directement utilisables.

In [ ]:
import numpy as np
from difmap_wrapper import DifmapSession

with DifmapSession() as session:
    session.observe(FITS_FILE)
    session.obs.select(pol=POL)
    data = session.obs.get_data()

u, v, amp, phase = data["u"], data["v"], data["amp"], data["phase"]
uv_dist = np.sqrt(u**2 + v**2) / 1e6

print(f"Visibilités        : {len(u)}")
print(f"Baseline max       : {uv_dist.max():.1f} Mλ")
print(f"Amplitude médiane  : {np.median(amp)*1000:.1f} mJy")
print(f"Amplitude max      : {amp.max()*1000:.1f} mJy")
print(f"Dispersion de phase: {phase.std():.1f}°")

---
### 2. Flagging programmatique

Avec subprocess, impossible de flaguer des visibilités sur un critère calculé en Python.
Avec le wrapper, on calcule le critère sur les arrays et on passe les indices directement.

In [ ]:
import matplotlib.pyplot as plt

SEUIL_AMP = 3.0  # Jy — on flag les visibilités aberrantes au-dessus de ce seuil

with DifmapSession() as session:
    session.observe(FITS_FILE)
    session.obs.select(pol=POL)
    data = session.obs.get_data()

    indices_aberrants = np.where(data["amp"] > SEUIL_AMP)[0]
    print(f"Visibilités à flaguer (amp > {SEUIL_AMP} Jy) : {len(indices_aberrants)}")

    if len(indices_aberrants) > 0:
        n_flagged = session.obs.flag_data(indices_aberrants)
        print(f"Flaguées : {n_flagged}")

    # Dirty map après flagging
    session.imager.mapsize(512, 0.1)
    session.imager.invert()
    session.vis.mapplot(title="Dirty Map après flagging", cmap="inferno", show=True)

---
### 3. Figures sur mesure

Le style PGPLOT de difmap est fixe (fond noir, vert, résolution basse).
Avec le wrapper on construit exactement la figure dont on a besoin.

In [ ]:
with DifmapSession() as session:
    session.observe(FITS_FILE)
    session.obs.select(pol=POL)
    data = session.obs.get_data()

u, v, amp = data["u"] / 1e6, data["v"] / 1e6, data["amp"] * 1000
uv_dist = np.sqrt(u**2 + v**2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# UV plot coloré par amplitude
sc = axes[0].scatter(u, v, c=amp, s=2, cmap="plasma", vmin=0, vmax=np.percentile(amp, 98))
axes[0].scatter(-u, -v, c=amp, s=2, cmap="plasma", vmin=0, vmax=np.percentile(amp, 98))
fig.colorbar(sc, ax=axes[0], label="Amplitude (mJy)")
axes[0].set_xlabel(r"U (M$\lambda$)"); axes[0].set_ylabel(r"V (M$\lambda$)")
axes[0].set_title("Couverture UV — couleur = amplitude")
axes[0].set_aspect("equal"); axes[0].invert_xaxis()
axes[0].grid(True, linestyle=":", alpha=0.4)

# Radplot avec médiane glissante
order = np.argsort(uv_dist)
axes[1].scatter(uv_dist, amp, s=1, alpha=0.3, color="navy")
axes[1].set_xlabel(r"Distance UV (M$\lambda$)"); axes[1].set_ylabel("Amplitude (mJy)")
axes[1].set_title("Amplitude vs distance UV")
axes[1].set_ylim(bottom=0); axes[1].grid(True, linestyle=":", alpha=0.4)

fig.suptitle("0003-066 — figures impossibles avec difmap seul", fontsize=13)
fig.tight_layout()
plt.show()